# Actividad — Comparación de métodos de selección de variables
**Aprendizaje Supervisado · Unidad 3 · Ingeniería y selección de variables**

**Qué haremos.** En la presentación vimos cómo *crear* variables y tres familias de métodos para *seleccionarlas*
(filtro, envoltura e integrado). Hoy los aplicarás al conjunto de **cáncer de mama de Wisconsin** (569 pacientes, 30 mediciones
de núcleos celulares, diagnóstico *maligno* o *benigno*) y compararás **con el mismo protocolo**: ¿cuántas variables bastan?
¿qué método elige mejor? ¿las listas de variables son estables? ¿qué pasa si seleccionas mal?

| Parte | Pregunta que responde | Tema de la presentación |
|---|---|---|
| 0–1 | ¿Qué datos tenemos, cuál es la línea base y cuánto dañan las variables inútiles? | Introducción |
| 2 | ¿Ayudan las variables del dominio y los polinomios? | A1–A8 |
| 3 | ¿Qué hacen la varianza, la correlación y la información mutua? | C1–C9 |
| 4 | ¿Qué variables sobreviven a la eliminación recursiva (RFE)? | D1–D5 |
| 5 | ¿Qué variables conserva Lasso? | E1–E5 |
| 6 | ¿Cómo se comparan los cuatro métodos, con la selección **dentro** del pipeline? | F1, F4 |
| 7 | ¿Eligen siempre las mismas variables? | F2 |
| 8 | ¿Cómo se ve (y se evita) una fuga de datos al seleccionar? | C9 |
| 9 | Recomendación final | Receta de siete pasos |

**Tiempo estimado:** 60–75 minutos de trabajo individual.

**Entregable:** este notebook ejecutado (con tus respuestas en las celdas ✏️) y el archivo `entregable_unidad3.csv`
con la tabla *método · k · exactitud de validación · sensibilidad · variables elegidas · comentario* de la Parte 6.

**Cómo trabajar.** Ejecuta las celdas en orden. Donde dice ✏️ hay una línea de código por completar o una pregunta;
escribe tus respuestas en la celda de texto que sigue a la pregunta, bajo «**Respuesta:**».
Todas las cifras de exactitud vienen de **validación cruzada con 5 particiones repetida 3 veces** sobre el conjunto de entrenamiento;
el conjunto de prueba se usa **una sola vez**, al final de la Parte 6.

> ⚠️ Es un conjunto médico: una variable que «ayuda» al modelo no es necesariamente una variable que un experto aceptaría.
> Por eso, además de la exactitud, medimos la **sensibilidad**: de los casos *malignos*, ¿qué fracción detecta el modelo?

In [ ]:
import warnings, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from functools import partial
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_validate, cross_val_score, RepeatedStratifiedKFold, StratifiedKFold
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures, FunctionTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.feature_selection import (VarianceThreshold, SelectKBest, f_classif, mutual_info_classif,
                                       RFE, RFECV, SelectFromModel)
from sklearn.metrics import make_scorer, recall_score, accuracy_score, confusion_matrix

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.3})
pd.set_option("display.width", 140)

# La información mutua se estima con vecinos y usa un poco de azar: fijamos la semilla para que los resultados se repitan
mi_clasif = partial(mutual_info_classif, random_state=0)

## Parte 0 — Los datos

El conjunto viene incluido en scikit-learn. Cada paciente tiene **10 medidas** del núcleo de las células de una biopsia
(radio, textura, perímetro, área, suavidad, compacidad, concavidad, puntos cóncavos, simetría y dimensión fractal),
cada una resumida de **tres formas**: media (`mean`), error estándar (`error`) y peor valor (`worst`). En total, 30 variables.
En `y`, **0 = maligno** y **1 = benigno**.

In [ ]:
bc = load_breast_cancer(as_frame=True)
X, y = bc.data, bc.target          # y: 0 = maligno, 1 = benigno

print(X.shape)
print(y.map({0: "maligno", 1: "benigno"}).value_counts().to_string())
X.iloc[:, :6].head()

## Parte 1 — División, línea base y el daño de las variables inútiles

Regla de la receta: lo primero es separar el conjunto de prueba, **estratificando** por el diagnóstico para conservar la proporción de casos malignos.
Después medimos una **línea base**: regresión logística con las 30 variables estandarizadas.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)    # ✏️ añade stratify=y
print("Entrenamiento:", X_train.shape, " Prueba:", X_test.shape)
print(f"Proporción de malignos -> entrenamiento: {100 * (y_train == 0).mean():.1f} %   prueba: {100 * (y_test == 0).mean():.1f} %")

# Protocolo de validación que usaremos SIEMPRE (el conjunto de prueba no se toca hasta el final de la Parte 6)
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=0)
sens_maligno = make_scorer(recall_score, pos_label=0)      # de los casos malignos, ¿cuántos detecta?


def evalua(pipe, Xd=None):
    # Devuelve (exactitud, sensibilidad a malignos), promedios de la validación cruzada, sobre el conjunto de entrenamiento
    Xd = X_train if Xd is None else Xd
    r = cross_validate(pipe, Xd, y_train, cv=cv, n_jobs=-1,
                       scoring={"exactitud": "accuracy", "sens": sens_maligno})
    return r["test_exactitud"].mean(), r["test_sens"].mean()


base = ...     # ✏️ Pipeline: StandardScaler → LogisticRegression(max_iter=3000)
acc0, sens0 = evalua(base)
print(f"Línea base (30 variables): exactitud = {acc0:.3f}   sensibilidad a malignos = {sens0:.3f}")

✏️ **Pregunta 1.1.** ¿Por qué estandarizamos antes de la regresión logística y por qué lo hacemos *dentro* del `Pipeline`?
En un diagnóstico médico, ¿por qué no basta con la exactitud? ¿Qué error es más grave: un maligno clasificado como benigno o al revés?

**Respuesta:**

In [ ]:
# ¿Cuánto daña agregar columnas que no significan nada? Añadimos ruido gaussiano puro y medimos de nuevo.
rng = np.random.default_rng(0)
filas = []
for n_ruido in [0, 50, 200]:
    Xr = X_train.copy()
    for j in range(n_ruido):
        Xr[f"ruido_{j}"] = rng.normal(size=len(Xr))
    for nombre, modelo in [("Regresión logística", make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000))),
                           ("KNN (k=5)", make_pipeline(StandardScaler(), KNeighborsClassifier(5)))]:
        filas.append({"columnas de ruido": n_ruido, "modelo": nombre, "exactitud": evalua(modelo, Xr)[0]})
print(pd.DataFrame(filas).pivot(index="columnas de ruido", columns="modelo", values="exactitud").round(3))

✏️ **Pregunta 1.2.** ¿Qué modelo sufre más con las columnas de ruido y por qué? ¿Qué te dice esto sobre la utilidad de **seleccionar** variables?

**Respuesta:**

## Parte 2 — Crear variables del dominio y polinomios  *(presentación A1–A8)*

Un patólogo no mira solo el tamaño del núcleo: mira qué tan **irregular** es. Un círculo perfecto cumple $P^2=4\pi A$ (perímetro al cuadrado entre área),
y cualquier irregularidad aumenta esa razón. Definimos la **circularidad** $P^2/(4\pi A)$ (vale 1 para un círculo y crece con las irregularidades)
y varias razones «peor valor entre valor medio». Las envolvemos en un `FunctionTransformer` para que sean un paso más del pipeline.

In [ ]:
def agrega_dominio(d):
    d = d.copy()
    d["circularidad media"] = d["mean perimeter"] ** 2 / (4 * np.pi * d["mean area"])
    d["circularidad peor"] = d["worst perimeter"] ** 2 / (4 * np.pi * d["worst area"])
    d["razon radio peor/medio"] = ...      # ✏️ worst radius entre mean radius
    d["razon textura peor/medio"] = ...    # ✏️ worst texture entre mean texture
    return d


nuevas = agrega_dominio(X_train).iloc[:, 30:]
print(nuevas.describe().loc[["mean", "min", "max"]].round(2).T)
print("\nCorrelación con el diagnóstico (|r|):")
print(nuevas.corrwith(y_train).abs().round(3).to_string())

In [ ]:
con_dominio = make_pipeline(FunctionTransformer(agrega_dominio), StandardScaler(), LogisticRegression(max_iter=3000))
acc1, sens1 = evalua(con_dominio)
print(f"30 variables originales     : exactitud = {acc0:.3f}   sensibilidad = {sens0:.3f}")
print(f"30 + 4 variables de dominio : exactitud = {acc1:.3f}   sensibilidad = {sens1:.3f}")

✏️ **Pregunta 2.1.** ¿Mejoraron las variables de dominio la exactitud? ¿Qué te dice la correlación de cada una con el diagnóstico?
¿Por qué una variable nueva puede no ayudar cuando el modelo ya tiene 30 variables correlacionadas entre sí?

**Respuesta:**

**Polinomios.** Ahora generamos potencias y productos con `PolynomialFeatures`. Primero contamos cuántas columnas aparecen y
después evaluamos la regresión logística sobre ellas.

In [ ]:
poly = ...     # ✏️ PolynomialFeatures(degree=2, include_bias=False).fit(X_train)
print("Columnas con grado 2 (30 variables):", poly.n_output_features_, "  (fórmula C(n+d, d) - 1 =", math.comb(32, 2) - 1, ")")
print("Columnas con interaction_only      :", PolynomialFeatures(degree=2, interaction_only=True, include_bias=False).fit(X_train).n_output_features_)
print("Columnas con grado 3               :", PolynomialFeatures(degree=3, include_bias=False).fit(X_train).n_output_features_)
print("Filas de entrenamiento             :", len(X_train))

for C_reg in [1.0, 0.01]:
    pipe = make_pipeline(StandardScaler(), PolynomialFeatures(degree=2, include_bias=False), StandardScaler(),
                         LogisticRegression(C=C_reg, max_iter=5000))
    a, s = evalua(pipe)
    print(f"Polinomio grado 2, C = {C_reg:<5}: exactitud = {a:.3f}   sensibilidad = {s:.3f}")

✏️ **Pregunta 2.2.** (a) ¿Cuántas columnas genera el grado 2 con 30 variables y cómo se compara con el número de filas? (b) ¿Qué ocurre con la exactitud?
¿Qué efecto tiene regularizar más (C pequeño)? (c) ¿Qué harías para usar polinomios sin perder control?

**Respuesta:**

## Parte 3 — Métodos de filtro  *(presentación C1–C9)*

Los filtros puntúan cada variable con una estadística sencilla, **sin entrenar un modelo**. Los probamos de uno en uno.

### 3.1 Umbral de varianza

In [ ]:
var = X_train.var().sort_values()
print("Varianzas más pequeñas:"); print(var.head(5).map("{:.2e}".format).to_string())
print("\nVarianzas más grandes:"); print(var.tail(3).map("{:.2e}".format).to_string())

vt = ...     # ✏️ VarianceThreshold(threshold=0.01).fit(X_train)
quedan = list(X_train.columns[vt.get_support()])
print(f"\nCon umbral 0.01 quedan {len(quedan)} de 30 variables (se descartan {30 - len(quedan)}).")

mi = pd.Series(mutual_info_classif(X_train, y_train, random_state=0), index=X_train.columns).sort_values(ascending=False)
top10_mi = list(mi.index[:10])
print("De las 10 variables con más información mutua con el diagnóstico, el filtro descarta:", [c for c in top10_mi if c not in quedan])

Xs_train = pd.DataFrame(StandardScaler().fit_transform(X_train), index=X_train.index, columns=X_train.columns)
print("Tras estandarizar, con el mismo umbral sobreviven:", int(VarianceThreshold(0.01).fit(Xs_train).get_support().sum()), "variables")

✏️ **Pregunta 3.1.** ¿Por qué el umbral de varianza descarta variables muy informativas sobre el diagnóstico? ¿Qué pasa si estandarizas antes y por qué?
¿Para qué sí sirve este filtro?

**Respuesta:**

### 3.2 Correlación con el objetivo y redundancia

In [ ]:
corr_y = X_train.corrwith(y_train).abs().sort_values(ascending=False)       # |r| de Pearson con el diagnóstico
print(corr_y.head(10).round(3).to_string())
print(f"\nVariables con |r| > 0.5: {(corr_y > 0.5).sum()}    con |r| > 0.7: {(corr_y > 0.7).sum()}    |r| mínimo: {corr_y.min():.3f}")

# Redundancia: pares de variables que dicen casi lo mismo
R = X_train.corr().abs()
pares = [(a, b, R.loc[a, b]) for i, a in enumerate(R.columns) for b in R.columns[i + 1:] if R.loc[a, b] > 0.9]
print(f"\nPares con |r| > 0.9: {len(pares)} de {30 * 29 // 2}")
for a, b, r in sorted(pares, key=lambda t: -t[2])[:5]:
    print(f"   {a:>22s}  ~  {b:<22s}  {r:.3f}")

In [ ]:
def poda_redundancia(Xd, yd, umbral=0.9):
    # Recorre las variables de mayor a menor |r| con el objetivo y conserva una solo si no es redundante con las ya elegidas
    relev = Xd.corrwith(yd).abs().sort_values(ascending=False)
    elegidas = []
    for c in relev.index:
        if ...:      # ✏️ ¿|correlación| de c con TODAS las elegidas es <= umbral?
            elegidas.append(c)
    return elegidas


sin_red = poda_redundancia(X_train, y_train)
print(f"Tras la poda quedan {len(sin_red)} variables:")
print(sin_red)
a, s = evalua(make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000)), X_train[sin_red])
print(f"\nExactitud con esas {len(sin_red)} variables: {a:.3f}   (línea base con 30: {acc0:.3f})")
print("Nota: la poda usó las etiquetas de TODO el entrenamiento; por eso no la incluimos en la comparación honesta de la Parte 6.")

✏️ **Pregunta 3.2.** (a) Si eligieras las 5 variables con mayor |r| con el diagnóstico, ¿qué tipo de variables obtendrías (mira la lista y los pares redundantes)?
¿Por qué eso es un problema? (b) ¿Qué lograste con la poda de redundancia y qué costo tuvo en exactitud?

**Respuesta:**

### 3.3 Información mutua frente a correlación

In [ ]:
comp = pd.DataFrame({"|r| con el diagnóstico": corr_y, "información mutua": mi}).sort_values("información mutua", ascending=False)
print(comp.head(10).round(3))

rho = spearmanr(comp["|r| con el diagnóstico"], comp["información mutua"])[0]
top10_r = set(corr_y.index[:10])
print(f"\nCorrelación de rangos (Spearman) entre ambos órdenes: {rho:.2f}")
print("Variables en el top 10 de ambos:", len(top10_r & set(top10_mi)), "de 10")
print("Solo en el top 10 de información mutua:", sorted(set(top10_mi) - top10_r))
print("Solo en el top 10 de |r|             :", sorted(top10_r - set(top10_mi)))

✏️ **Pregunta 3.3.** ¿Se parecen los órdenes de información mutua y de correlación en estos datos? ¿Por qué? Da un ejemplo de relación en la que la información mutua
detectaría algo que la correlación de Pearson no.

**Respuesta:**

### 3.4 ¿Cuántas variables conservar? `SelectKBest` **dentro** del pipeline

In [ ]:
ks = [1, 2, 3, 5, 8, 10, 15, 20, 30]
curvas = {}
for nombre, score in [("información mutua", mi_clasif), ("ANOVA F (≈ correlación)", f_classif)]:
    curvas[nombre] = []
    for k in ks:
        pipe = ...     # ✏️ Pipeline: StandardScaler → SelectKBest(score, k=k) → LogisticRegression(max_iter=3000)
        curvas[nombre].append(evalua(pipe)[0])

tabla_k = pd.DataFrame(curvas, index=ks).round(3); tabla_k.index.name = "k"
print(tabla_k.T.to_string())
fig, ax = plt.subplots(figsize=(6.5, 3.2))
for nombre, vals in curvas.items():
    ax.plot(ks, vals, marker="o", label=nombre)
ax.axhline(acc0, color="gray", ls="--", lw=1, label="30 variables")
ax.set_xlabel("k = variables que se conservan"); ax.set_ylabel("exactitud (validación)"); ax.legend(); plt.tight_layout(); plt.show()

✏️ **Pregunta 3.4.** ¿Con cuántas variables ($k$) un filtro iguala a la línea base? ¿Cómo elegirías $k$ de forma honesta? ¿Por qué es importante que `SelectKBest`
esté **dentro** del `Pipeline` y no antes de la validación cruzada?

**Respuesta:**

## Parte 4 — Envoltura: eliminación recursiva de variables (RFE)  *(presentación D1–D5)*

RFE entrena el modelo, quita la variable con menor |coeficiente|, y repite. Se usa sobre variables **estandarizadas** para que los coeficientes sean comparables.
Usaremos `Xs_train` (el entrenamiento ya estandarizado, definido en la Parte 3).

In [ ]:
rfe = ...     # ✏️ RFE(LogisticRegression(max_iter=3000), n_features_to_select=10, step=1).fit(Xs_train, y_train)
vars_rfe10 = list(X_train.columns[rfe.support_])
print("RFE con k = 10 elige:")
print(vars_rfe10)

# Orden de eliminación completo: RFE hasta dejar una sola variable (ranking 1 = la que sobrevive hasta el final)
orden = pd.Series(RFE(LogisticRegression(max_iter=3000), n_features_to_select=1, step=1).fit(Xs_train, y_train).ranking_,
                  index=X_train.columns).sort_values()
print("\nÚltimas en sobrevivir :", list(orden.index[:5]))
print("Primeras en eliminarse:", list(orden.index[-5:][::-1]))

In [ ]:
# RFECV: la validación cruzada decide cuántas variables conservar
rfecv = ...     # ✏️ RFECV(LogisticRegression(max_iter=3000), step=1, cv=StratifiedKFold(5, shuffle=True, random_state=0), scoring="accuracy").fit(Xs_train, y_train)
curva = rfecv.cv_results_["mean_test_score"]
print("Número óptimo de variables según RFECV:", rfecv.n_features_, f"(exactitud {curva.max():.3f})")
for k in [1, 3, 5, 8, 10, 15, 30]:
    print(f"   con {k:2d} variables: {curva[k - 1]:.3f}")

fig, ax = plt.subplots(figsize=(6.5, 3.0))
ax.plot(range(1, 31), curva, marker="o", ms=3)
ax.axvline(rfecv.n_features_, color="C1", ls="--"); ax.set_xlabel("variables que quedan"); ax.set_ylabel("exactitud (validación)")
plt.tight_layout(); plt.show()

✏️ **Pregunta 4.1.** (a) ¿Qué tipo de variables sobreviven hasta el final y cuáles se eliminan primero? ¿Hay medidas de textura entre las que sobreviven?
(b) ¿Cuántas variables recomienda RFECV y cuántas elegirías tú viendo la curva? (c) ¿Por qué RFE reentrena en cada ronda en lugar de ordenar una sola vez?

**Respuesta:**

## Parte 5 — Integrado: selección con Lasso  *(presentación E1–E5)*

La regresión logística con penalización L1 lleva coeficientes a cero. El parámetro `C` $=1/\alpha$ controla la fuerza:
**menor C, más penalización**. Trazamos la trayectoria completa y elegimos `C` por validación cruzada.

In [ ]:
Cs = np.logspace(-2.3, 1.0, 30)
coefs, nz, accs = [], [], []
for C_ in Cs:
    m = ...     # ✏️ LogisticRegression(penalty="l1", solver="liblinear", random_state=0, C=C_, max_iter=5000).fit(Xs_train, y_train)
    coefs.append(m.coef_[0]); nz.append(int((m.coef_[0] != 0).sum()))
    accs.append(cross_val_score(LogisticRegression(penalty="l1", solver="liblinear", random_state=0, C=C_, max_iter=5000), Xs_train, y_train,
                                cv=StratifiedKFold(5, shuffle=True, random_state=0)).mean())
coefs = np.array(coefs)
k_best = int(np.argmax(accs))
print(f"Mejor C por validación: {Cs[k_best]:.3f} -> {nz[k_best]} variables con coeficiente distinto de cero, exactitud {accs[k_best]:.3f}")
for i in range(0, 30, 6):
    print(f"   C = {Cs[i]:7.3f}: {nz[i]:2d} variables activas, exactitud {accs[i]:.3f}")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
ax[0].plot(np.log10(Cs), coefs, lw=1); ax[0].axhline(0, color="k", lw=0.6); ax[0].axvline(np.log10(Cs[k_best]), color="C1", ls="--")
ax[0].set_xlabel("log10 C  (menos penalización →)"); ax[0].set_ylabel("coeficiente")
ax[1].plot(np.log10(Cs), accs, color="C0"); ax[1].set_ylabel("exactitud (validación)", color="C0"); ax[1].set_xlabel("log10 C")
ax2 = ax[1].twinx(); ax2.plot(np.log10(Cs), nz, color="C1"); ax2.set_ylabel("variables con coeficiente ≠ 0", color="C1"); ax2.grid(False)
plt.tight_layout(); plt.show()

In [ ]:
# SelectFromModel: se queda con las variables de coeficiente distinto de cero
sfm = ...     # ✏️ SelectFromModel(LogisticRegression(penalty="l1", solver="liblinear", random_state=0, C=0.3, max_iter=5000)).fit(Xs_train, y_train)
vars_lasso = list(X_train.columns[sfm.get_support()])
print(f"Lasso con C = 0.3 conserva {len(vars_lasso)} variables:")
print(vars_lasso)
print("\nEn común con RFE (k=10):", sorted(set(vars_lasso) & set(vars_rfe10)))

✏️ **Pregunta 5.1.** (a) Describe qué ocurre con los coeficientes cuando disminuye `C`. (b) ¿Cuántas variables activas deja el mejor C y qué pasa con la exactitud si te quedas con solo 4 variables activas?
(c) ¿Qué ocurriría si **no** estandarizaras antes de Lasso? (d) ¿Cuántas variables comparten Lasso y RFE y qué sugiere eso?

**Respuesta:**

## Parte 6 — Comparación de los cuatro métodos con el mismo protocolo  *(presentación F1, F4)*

Ahora ponemos los cuatro métodos en igualdad de condiciones: **StandardScaler → selector → regresión logística**, con la selección **dentro**
del `Pipeline` (se rehace en cada partición). Para cada método y número de variables $k$ medimos exactitud **y** sensibilidad a casos malignos.

In [ ]:
def lr(): return LogisticRegression(max_iter=3000)

METODOS = ["Filtro: información mutua", "Filtro: ANOVA F", "Envoltura: RFE", "Integrado: Lasso"]


def selector(metodo, k):
    if metodo == "Filtro: información mutua":
        return SelectKBest(mi_clasif, k=k)
    if metodo == "Filtro: ANOVA F":
        return SelectKBest(f_classif, k=k)
    if metodo == "Envoltura: RFE":
        return ...     # ✏️ RFE(lr(), n_features_to_select=k, step=1)
    if metodo == "Integrado: Lasso":
        return ...     # ✏️ SelectFromModel(LogisticRegression(penalty="l1", solver="liblinear", random_state=0, C=0.3, max_iter=5000), max_features=k, threshold=-np.inf)


ks = [3, 5, 8, 10, 15, 20, 30]
filas = []
for metodo in METODOS:
    for k in ks:
        pipe = Pipeline([("sc", StandardScaler()), ("sel", selector(metodo, k)), ("lr", lr())])
        a, s = evalua(pipe)
        filas.append({"método": metodo, "k": k, "exactitud": a, "sensibilidad": s})
res = pd.DataFrame(filas)
print("Exactitud de validación (línea base con 30 variables: %.3f)" % acc0)
print(res.pivot(index="método", columns="k", values="exactitud").round(3).to_string())
print("\nSensibilidad a malignos (línea base: %.3f)" % sens0)
print(res.pivot(index="método", columns="k", values="sensibilidad").round(3).to_string())

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4), sharex=True)
for metodo in METODOS:
    d = res[res["método"] == metodo]
    ax[0].plot(d["k"], d["exactitud"], marker="o", ms=4, label=metodo)
    ax[1].plot(d["k"], d["sensibilidad"], marker="o", ms=4, label=metodo)
ax[0].axhline(acc0, color="gray", ls="--", lw=1); ax[1].axhline(sens0, color="gray", ls="--", lw=1)
ax[0].set_ylabel("exactitud (validación)"); ax[1].set_ylabel("sensibilidad a malignos (validación)")
for a_ in ax: a_.set_xlabel("k = variables que se conservan")
ax[0].legend(fontsize=7); plt.tight_layout(); plt.show()

✏️ **Pregunta 6.1.** (a) Con pocas variables ($k$ entre 5 y 10), ¿qué métodos rinden mejor y qué tienen en común sus listas? (b) ¿A partir de qué $k$ se igualan todos con la línea base y por qué?
(c) ¿Cambia la conclusión si miras la sensibilidad en vez de la exactitud?

**Respuesta:**

**¿Qué variables elige cada método?** Ajustamos cada uno con $k=8$ sobre todo el entrenamiento y listamos las elegidas.
Luego armas la **tabla entregable**.

In [ ]:
K_TABLA = 8
elegidas = {}
for metodo in METODOS:
    sel = selector(metodo, K_TABLA).fit(Xs_train, y_train)
    elegidas[metodo] = list(X_train.columns[sel.get_support()])
    print(f"{metodo:26s}", elegidas[metodo])

# Comentarios para cada fila (✏️ escribe el tuyo en el alumno)
comentarios = {
    "Línea base (30 variables)": "✏️ escribe tu comentario",
    "Filtro: información mutua": "✏️ escribe tu comentario",
    "Filtro: ANOVA F": "✏️ escribe tu comentario",
    "Envoltura: RFE": "✏️ escribe tu comentario",
    "Integrado: Lasso": "✏️ escribe tu comentario",
}

In [ ]:
filas = [{"método": "Línea base (30 variables)", "k": 30, "exactitud_cv": round(acc0, 3), "sensibilidad_cv": round(sens0, 3),
          "variables": "todas", "comentario": comentarios["Línea base (30 variables)"]}]
for metodo in METODOS:
    f_ = res[(res["método"] == metodo) & (res["k"] == K_TABLA)].iloc[0]
    filas.append({"método": metodo, "k": K_TABLA, "exactitud_cv": round(f_["exactitud"], 3), "sensibilidad_cv": round(f_["sensibilidad"], 3),
                  "variables": "; ".join(elegidas[metodo]), "comentario": comentarios[metodo]})
entregable = pd.DataFrame(filas)
entregable.to_csv("entregable_unidad3.csv", index=False)
entregable[["método", "k", "exactitud_cv", "sensibilidad_cv", "comentario"]]

### Elegir un modelo y usar el conjunto de prueba **una sola vez**

Con lo que viste, elige un método y un $k$ (la regla: el modelo más **pequeño** cuya validación quede cerca de la mejor). Ajústalo con todo el entrenamiento
y evalúalo en el conjunto de prueba, junto con la línea base.

In [ ]:
metodo_elegido, k_elegido = ..., ...     # ✏️ por ejemplo ("Envoltura: RFE", 10)
final = Pipeline([("sc", StandardScaler()), ("sel", selector(metodo_elegido, k_elegido)), ("lr", lr())]).fit(X_train, y_train)
baseline = make_pipeline(StandardScaler(), lr()).fit(X_train, y_train)

for nombre, mod in [("Línea base (30)", baseline), (f"{metodo_elegido}, k={k_elegido}", final)]:
    pred = mod.predict(X_test)
    cm = confusion_matrix(y_test, pred, labels=[0, 1])
    print(f"{nombre:28s} exactitud en prueba = {accuracy_score(y_test, pred):.3f}   sensibilidad a malignos = {recall_score(y_test, pred, pos_label=0):.3f}"
          f"   (malignos perdidos: {cm[0, 1]} de {cm[0].sum()}; benignos con falsa alarma: {cm[1, 0]} de {cm[1].sum()})")
print("\nVariables del modelo final:", list(X_train.columns[final[:-1].named_steps['sel'].get_support()]))

✏️ **Pregunta 6.2.** ¿Qué método y $k$ elegiste y por qué? ¿Se parece la exactitud en prueba a la de validación? Con solo 114 pacientes en prueba, ¿cuánta
diferencia entre dos modelos te parecería real? ¿Un paciente menos o más detectado cambia mucho la sensibilidad?

**Respuesta:**

## Parte 7 — Estabilidad de la selección  *(presentación F2)*

Una lista de «variables importantes» que cambia con cada muestra no sirve para explicar nada. Repetimos la selección de $k=5$ variables
con **30 remuestras bootstrap** del entrenamiento y contamos con qué frecuencia se elige cada variable.

In [ ]:
B, k5 = 30, 5
rng = np.random.default_rng(0)
freq = {m: pd.Series(0.0, index=X_train.columns) for m in METODOS}
for b in range(B):
    idx = rng.choice(len(X_train), len(X_train), replace=True)
    Xb, yb = X_train.iloc[idx], y_train.iloc[idx]
    Xbs = pd.DataFrame(StandardScaler().fit_transform(Xb), columns=X_train.columns)
    for m in METODOS:
        freq[m][X_train.columns[selector(m, k5).fit(Xbs, yb).get_support()]] += 1 / B
F = pd.DataFrame(freq)
tabla_est = F.loc[F.max(axis=1).sort_values(ascending=False).index[:12]].round(2)
print(tabla_est.to_string())
resumen = pd.DataFrame({"frecuencia media de las 5 más elegidas": [np.sort(F[m].values)[::-1][:k5].mean() for m in METODOS],
                        "variables distintas que alguna vez se eligieron": [int((F[m] > 0).sum()) for m in METODOS]}, index=METODOS)
print(); print(resumen.round(2).to_string())

✏️ **Pregunta 7.1.** ¿Qué métodos son más estables y cuáles reparten su elección entre muchas variables? ¿Por qué ocurre eso con RFE y Lasso en estos datos?
Si tuvieras que **explicarle a un médico** qué variables importan, ¿cómo usarías esta tabla?

**Respuesta:**

## Parte 8 — Una fuga de datos al seleccionar  *(presentación C9)*

Seleccionar variables usa las etiquetas. Lo comprobamos con un caso en el que **no hay nada que aprender**: 60 filas, 1 000 variables de ruido puro y etiquetas al azar.
Un modelo honesto debe acertar cerca de 0.50.

In [ ]:
rng = np.random.default_rng(0)
Xr = rng.normal(size=(60, 1000)); yr = rng.integers(0, 2, 60)
cv_r = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=0)

# MAL: elegir las 10 «mejores» con TODAS las etiquetas y después validar
Xsel = SelectKBest(f_classif, k=10).fit_transform(Xr, yr)
mal = cross_val_score(LogisticRegression(), Xsel, yr, cv=cv_r).mean()

# BIEN: el selector es un paso del Pipeline (se rehace en cada partición)
pipe_bien = ...     # ✏️ Pipeline: SelectKBest(f_classif, k=10) → LogisticRegression()
bien = cross_val_score(pipe_bien, Xr, yr, cv=cv_r).mean()
print(f"Selección ANTES de validar (fuga): exactitud = {mal:.3f}")
print(f"Selección DENTRO del pipeline    : exactitud = {bien:.3f}   (azar = 0.50)")

✏️ **Pregunta 8.1.** ¿Por qué la selección fuera de la validación da una exactitud tan alta con datos que son puro ruido? ¿Cuál de las dos cifras es creíble?
¿Qué otras operaciones de esta unidad tendrían el mismo problema si se hicieran antes de dividir?

**Respuesta:**

## Parte 9 — Recomendación final

✏️ **Pregunta 9.1.** Redacta una recomendación breve (6 a 8 líneas) para el equipo médico: qué método y cuántas variables propones, con qué exactitud y sensibilidad esperada
(**validación**, no prueba), qué variables componen el modelo, qué cuidados tomaste para no engañarte (fuga, estabilidad) y qué validación necesitarías de un experto antes de usarlo.

**Respuesta:**

---
**Reto opcional.** Agrega `ElasticNet` a la comparación (`LogisticRegression(penalty="elasticnet", solver="saga", random_state=0, l1_ratio=0.5)` dentro de `SelectFromModel`)
y compara su lista de variables con la de Lasso con $k=8$. ¿Qué esperarías para variables correlacionadas?

In [ ]:
# Reto opcional
def selector_en(k):
    return SelectFromModel(LogisticRegression(penalty="elasticnet", solver="saga", random_state=0, l1_ratio=0.5, C=0.3, max_iter=5000),
                           max_features=k, threshold=-np.inf)

pipe_en = Pipeline([("sc", StandardScaler()), ("sel", selector_en(8)), ("lr", lr())])
a, s = evalua(pipe_en)
print(f"Elastic Net (l1_ratio=0.5), k=8: exactitud = {a:.3f}   sensibilidad = {s:.3f}")
sel_en = selector_en(8).fit(Xs_train, y_train)
print("Variables:", list(X_train.columns[sel_en.get_support()]))